# Web Scraping using Requests and BeautifulSoup

In [1]:
!pip install requests
!pip install beautifulsoup4

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

In [3]:
# Send a GET request and parse the HTML
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [4]:
# Extract book containers (each book is in an <article> tag with class "product_pod")
books = soup.find_all('article', class_='product_pod')
print(f"Found {len(books)} books on the page.")

Found 20 books on the page.


In [5]:
# Loop through each book and extract title, price, and availability
for book in books:
    # Title is in <h3> > <a> tag and the title attribute
    title = book.h3.a['title']

    # Price is in <p class="price_color">
    price = book.find('p', class_='price_color').text.strip()

    # Availability is in <p class="instock availability">
    availability = book.find('p', class_='instock availability').text.strip()

    print(f"Title: {title}\nPrice: {price}\nAvailability: {availability}\n{'-'*50}")


Title: A Light in the Attic
Price: Â£51.77
Availability: In stock
--------------------------------------------------
Title: Tipping the Velvet
Price: Â£53.74
Availability: In stock
--------------------------------------------------
Title: Soumission
Price: Â£50.10
Availability: In stock
--------------------------------------------------
Title: Sharp Objects
Price: Â£47.82
Availability: In stock
--------------------------------------------------
Title: Sapiens: A Brief History of Humankind
Price: Â£54.23
Availability: In stock
--------------------------------------------------
Title: The Requiem Red
Price: Â£22.65
Availability: In stock
--------------------------------------------------
Title: The Dirty Little Secrets of Getting Your Dream Job
Price: Â£33.34
Availability: In stock
--------------------------------------------------
Title: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Price: Â£17.93
Availability: In stock
-------------------------

In [6]:
import csv

def scrape_and_save_to_csv(start_page=1, end_page=2, filename='books.csv'):
    base_url = "https://books.toscrape.com/catalogue/page-{}.html"
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Title', 'Price', 'Availability'])

        for page_num in range(start_page, end_page+1):
            url = base_url.format(page_num)
            print(f"Scraping: {url}")
            response = requests.get(url)
            soup = BeautifulSoup(response.text, 'html.parser')
            books = soup.find_all('article', class_='product_pod')

            for book in books:
                title = book.h3.a['title']
                price = book.find('p', class_='price_color').text.strip()
                availability = book.find('p', class_='instock availability').text.strip()
                writer.writerow([title, price, availability])


In [7]:
# Save pages 1 to 3 into CSV
scrape_and_save_to_csv(1, 3)

Scraping: https://books.toscrape.com/catalogue/page-1.html
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html


In [8]:
import pandas as pd

df = pd.read_csv("books.csv")
df

,Title,Price,Availability
0,A Light in the Attic,Â£51.77,In stock
1,Tipping the Velvet,Â£53.74,In stock
2,Soumission,Â£50.10,In stock
3,Sharp Objects,Â£47.82,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock
5,The Requiem Red,Â£22.65,In stock
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock
9,The Black Maria,Â£52.15,In stock


# Web Scraping using Scrapy

In [9]:
!pip install scrapy

In [10]:
!scrapy version

Scrapy 2.11.1


In [11]:
# Create project folder and navigate into it
!scrapy startproject books_scraper
%cd books_scraper

New Scrapy project 'books_scraper', using template directory 'C:\ProgramData\anaconda3\Lib\site-packages\scrapy\templates\project', created in:
    D:\1-AI_data_science\hands-on-experience\week4-2-june\day-3\books_scraper

You can start your first spider with:
    cd books_scraper
    scrapy genspider example example.com
D:\1-AI_data_science\hands-on-experience\week4-2-june\day-3\books_scraper


In [12]:
# Create the spider file manually
spider_code = """
import scrapy

class BooksSpider(scrapy.Spider):
    name = 'books'
    start_urls = ['https://books.toscrape.com/']

    def parse(self, response):
        for book in response.css('article.product_pod'):
            yield {
                'title': book.css('h3 a::attr(title)').get(),
                'price': book.css('p.price_color::text').get(),
                'availability': ''.join(book.css('p.instock.availability::text').getall()).strip()
            }

        next_page = response.css('li.next a::attr(href)').get()
        if next_page:
            yield response.follow(next_page, self.parse)
"""

# Save spider file
with open("books_scraper/spiders/books.py", "w") as f:
    f.write(spider_code)

In [13]:
# Run and save to CSV
!scrapy crawl books -o books.csv

2025-08-02 07:48:05 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: books_scraper)
2025-08-02 07:48:05 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)], pyOpenSSL 24.2.1 (OpenSSL 3.0.16 11 Feb 2025), cryptography 43.0.0, Platform Windows-10-10.0.19045-SP0
2025-08-02 07:48:05 [scrapy.addons] INFO: Enabled addons:
[]
2025-08-02 07:48:05 [asyncio] DEBUG: Using selector: SelectSelector
2025-08-02 07:48:05 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2025-08-02 07:48:05 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.windows_events._WindowsSelectorEventLoop
2025-08-02 07:48:05 [scrapy.extensions.telnet] INFO: Telnet Password: 8dbbbaf9b825060f
2025-08-02 07:48:06 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats'

In [14]:
import pandas as pd

df = pd.read_csv("books.csv")
df

EmptyDataError: No columns to parse from file

In [ ]:
!scrapy runspider books_scraper/spiders/books2.py -o books_clean.csv > /dev/null 2>&1

In [ ]:
import pandas as pd

df = pd.read_csv("books_clean.csv")
df

In [15]:
import requests

url = "https://api.coindesk.com/v1/bpi/currentprice.json"
response = requests.get(url)
data = response.json()

print(data['bpi']['USD']['rate'])  # Bitcoin price in USD

ConnectionError: HTTPSConnectionPool(host='api.coindesk.com', port=443): Max retries exceeded with url: /v1/bpi/currentprice.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001DE74725CD0>: Failed to resolve 'api.coindesk.com' ([Errno 11001] getaddrinfo failed)"))

In [16]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN

# Create a presentation
prs = Presentation()

# Theme colors
LIGHT_BLUE = RGBColor(91, 155, 213)
WHITE = RGBColor(255, 255, 255)

# Helper: Add slide with title and optional content
def add_slide(title, content=None, is_code=False):
    slide_layout = prs.slide_layouts[1]  # Title + Content
    slide = prs.slides.add_slide(slide_layout)

    # Title
    slide.shapes.title.text = title
    slide.shapes.title.text_frame.paragraphs[0].font.size = Pt(32)
    slide.shapes.title.text_frame.paragraphs[0].font.color.rgb = LIGHT_BLUE

    # Content
    if content:
        if is_code:
            # Add rounded rectangle for code
            left, top, width, height = Inches(1), Inches(2), Inches(8), Inches(3)
            shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left, top, width, height)
            shape.fill.solid()
            shape.fill.fore_color.rgb = WHITE
            shape.line.color.rgb = LIGHT_BLUE
            tf = shape.text_frame
            p = tf.paragraphs[0]
            p.text = content
            p.font.name = "Courier New"
            p.font.size = Pt(12)
            p.font.color.rgb = RGBColor(0, 0, 0)
        else:
            slide.placeholders[1].text = content

# Slide 1: Title (placeholder for image)
title_slide_layout = prs.slide_layouts[6]  # Blank
slide = prs.slides.add_slide(title_slide_layout)
left, top, width, height = Inches(0), Inches(0), Inches(10), Inches(7.5)
# Placeholder rectangle for title image
shape = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
shape.fill.solid()
shape.fill.fore_color.rgb = RGBColor(200, 200, 200)  # Grey placeholder
shape.line.fill.background()

# Title text on top of image placeholder
textbox = slide.shapes.add_textbox(Inches(1), Inches(2.5), Inches(8), Inches(2))
tf = textbox.text_frame
p = tf.paragraphs[0]
p.text = "Mastering APIs, Databases & Web Scraping"
p.font.size = Pt(36)
p.font.bold = True
p.font.color.rgb = WHITE
p.alignment = PP_ALIGN.CENTER

p2 = tf.add_paragraph()
p2.text = "From Data Access to Data Extraction"
p2.font.size = Pt(20)
p2.font.color.rgb = WHITE
p2.alignment = PP_ALIGN.CENTER

# Example slides
add_slide("What is an API?", "An API (Application Programming Interface) allows applications to communicate...")
add_slide("Types of APIs", "REST, SOAP, GraphQL, gRPC...")
add_slide("Python: Simple API Call", "import requests\nresponse = requests.get('https://api.example.com/data')\nprint(response.json())", is_code=True)

# Placeholder for diagram
diagram_slide = prs.slides.add_slide(prs.slide_layouts[6])
shape = diagram_slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(1), Inches(1.5), Inches(8), Inches(4.5))
shape.fill.solid()
shape.fill.fore_color.rgb = RGBColor(220, 230, 241)  # Light placeholder
shape.line.color.rgb = LIGHT_BLUE
diagram_slide.shapes.add_textbox(Inches(1), Inches(0.5), Inches(8), Inches(1)).text = "Diagram Placeholder"

# Closing slide
closing_slide = prs.slides.add_slide(prs.slide_layouts[6])
textbox = closing_slide.shapes.add_textbox(Inches(2.5), Inches(3), Inches(5), Inches(1.5))
p = textbox.text_frame.paragraphs[0]
p.text = "Thank You"
p.font.size = Pt(36)
p.font.bold = True
p.font.color.rgb = LIGHT_BLUE
p.alignment = PP_ALIGN.CENTER

# Save presentation
prs.save("APIs_Databases_WebScraping_Presentation.pptx")
print("Presentation created: APIs_Databases_WebScraping_Presentation.pptx")


Presentation created: APIs_Databases_WebScraping_Presentation.pptx
